# Práctica · ETL frente a ELT en el pipeline del acueducto

**IFPN0025 · Big Data e Ingeniería de Datos** · Universidad Ean · Módulo 2

**Qué hace este cuaderno.** Toma el trabajo que alimenta hoy el tablero financiero (extrae la cruda, descarta lo dudoso, agrega y guarda solo el resultado: ETL) y lo refactoriza a una ruta ELT sobre la capa de preparación `refinada_spark/` (guarda todo con una bandera y transforma después, con SQL). Comprueba que ambas rutas dan el **mismo resultado**, mide **qué lee y qué escribe cada una** con una métrica de bytes declarada, define **un contrato por frontera** que aprueba con los datos y falla donde debe ante una alteración inyectada, y **reprocesa una ventana de días** tocando solo esas particiones. Termina en un documento de decisión ETL/ELT.

| Nivel | Nombre | Dónde | Tiempo | Qué produce |
|---|---|---|---|---|
| 1 | Guiado | En clase | 45 min | Contraste con la solución de referencia de la lectura de planes; sesión; el ETL del tablero medido; el producto ELT por SQL y sus dos reconciliaciones; el segundo producto; el cambio de regla; el reproceso dinámico y estático; los contratos y la copia alterada; `comparacion_nivel1.csv` |
| 2 | Aplicado | Autónomo · obligatorio | 75 min dentro de las 3 h de la tarea | Lo mismo sobre la ampliación del equipo, con cuatro piezas que escribe cada equipo (ventana, SQL, contratos, hipótesis): `comparacion.csv`, `bitacora_capas.jsonl`, `evidencia_nivel2_<equipo>.md` y la etapa `t18` aprobada |
| 3 | Frontera | Autónomo · opcional | Fuera de las 3 h | 3a: qué no debe aterrizar. 3b: el mismo producto como modelo de dbt |
| · | Reto de diseño | Autónomo · opcional | 30 min | Ficha de la capa para la dirección financiera y operaciones |

**Cómo usarlo.** Ejecuten las celdas en orden. El Nivel 1 está resuelto para seguirlo en clase. En el Nivel 2 encontrarán celdas con `# TODO` y `NotImplementedError` que deben completar (reemplacen la función entera, docstring incluida, y borren el `raise`); después de cada ejercicio hay una celda de autoverificación con `assert`. Las decisiones de ingeniería de cada nivel están en la guía; aquí se documentan en las celdas de texto marcadas como **Su decisión**. Si reinician el núcleo, ejecuten de nuevo la configuración común, los ayudantes y la celda de la sesión.

**Entorno.** Perfil `s18` del paquete en marcha (`docker compose --profile s18 up -d`, servicio `healthy`), con este cuaderno guardado en la carpeta `practica/` y abierto desde JupyterLab (http://localhost:8888). Python 3.11, Java 11, PySpark 3.5.3, pandas y pyarrow ya vienen en la imagen. Los datos ya vienen generados con semilla fija. La sesión usa `local[4]` y `spark.driver.memory=1g`. Esperen un pico de memoria del contenedor de 1,7 a 2,3 GiB netos con el cuaderno completo (el valor más alto, si ejecutan también el Nivel 3b; el límite del contenedor es 3 GiB): abran una sola sesión de Spark a la vez. Los tiempos de reloj son orientativos y varían hasta ×3 según la carga del equipo; lo estable son los archivos, las filas, los bytes y los hashes.

**La regla de la práctica.** Una comparación entre dos enfoques solo vale si ambos dan el mismo resultado y si se declara qué se midió. Ninguna conclusión se apoya en tiempos; se apoya en bytes y archivos leídos y escritos, en filas descartadas que se pueden contar y en que el resultado sea idéntico.

### 0. Configuración común

La celda localiza el paquete, importa los módulos del proyecto (`elt_s18`, `etl_tablero`, `metricas_capas`, `refinada_spark`, `verificar_etapa`) y calcula el hash del lago heredado: al terminar, `datalake/` debe seguir idéntico.

In [1]:
# Configuracion comun: importaciones, rutas del paquete y los modulos del proyecto (los datos ya vienen generados con semilla fija)
import hashlib, importlib, json, os, re, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

EQUIPO = os.environ.get("EQUIPO", "equipo-1")   # cambien por su identificador (equipo-1 a equipo-6)
RAIZ = Path.cwd().parent if Path.cwd().name == "practica" else Path.cwd()   # carpeta raiz del paquete
sys.path.insert(0, str(RAIZ / "pipeline"))
import elt_s18 as E                 # ejecutor de capas: SQL, contratos, reproceso, bitacora
import etl_tablero as T             # el trabajo heredado del tablero (transforma antes de guardar)
import metricas_capas as M          # lo que cada trabajo leyo y escribio
import refinada_spark as R          # la capa de preparacion (se usa para reconstruir la copia alterada)
import verificar_etapa as V         # verificacion por etapa (hash del lago, etapa t18)

LAGO_BASE = RAIZ / "datalake"                 # base comun: 200 medidores, 576 000 lecturas
LAGO_EQUIPO = RAIZ / "datalake_equipo"        # ampliacion de su equipo
REF_SQL = RAIZ / "referencia_s18" / "sql"             # SQL resuelto que se usa en el Nivel 1
REF_CONTRATOS = RAIZ / "referencia_s18" / "contratos" # contratos resueltos que se usan en el Nivel 1
BITACORA = "bitacora_capas.jsonl"
DIA_FIJO = "2024-02-15"                        # dia fijo del Nivel 1
VENTANA_N1 = ("2024-02-10", "2024-02-15")      # los 6 dias que terminan en el dia fijo
HASHES_BASE_ANTES = {"lago": V.hash_arbol_lago(LAGO_BASE), "preparacion": V.hash_arbol_capa(LAGO_BASE / "refinada_spark")}
print("Equipo:", EQUIPO, "| carpeta raiz:", RAIZ)
print("Hash del lago heredado (datalake/):", HASHES_BASE_ANTES["lago"][:16], "...")

Equipo: equipo-1 | carpeta raiz: /proyecto
Hash del lago heredado (datalake/): adac5febf8e7e3f5 ...


**Ayudantes.** `ultima` busca la última línea de la bitácora de capas que cumple unos criterios, `mostrar` imprime una medición, `leer_salida_etl` lee el resultado del ETL y `correr_etl` ejecuta el trabajo heredado con la sesión ya abierta.

In [2]:
# Ayudantes del cuaderno: leer la bitacora, mostrar una medicion y convertir tablas a Markdown
def ultima(volumen, **criterios):
    """Ultima linea de la bitacora de capas del volumen que cumple los criterios (accion, enfoque, producto...)."""
    sel = [b for b in E.leer_bitacora(RAIZ, volumen, BITACORA) if all(b.get(k) == v for k, v in criterios.items())]
    return sel[-1] if sel else {}

def mostrar(b):
    """Lo esencial de una medicion: archivos y bytes leidos (metrica declarada), bytes escritos, etapas y mezcla."""
    print(f"{b['enfoque']:15s} {b['producto']:8s} archivos leidos {b['archivos_leidos']:4d} · bytes leidos {b['bytes_leidos']:>11,} · "
          f"escritos {b['bytes_escritos']:>9,} B en {b['archivos_escritos']} archivo(s) · etapas {b['etapas']} · tareas {b['tareas']} · mezcla {b['bytes_mezcla']:,} B")

def a_markdown(tabla):
    """DataFrame de pandas a tabla Markdown (sin dependencias adicionales)."""
    filas = ["| " + " | ".join(str(c) for c in tabla.columns) + " |", "|" + "---|" * len(tabla.columns)]
    filas += ["| " + " | ".join(str(v) for v in f) + " |" for f in tabla.itertuples(index=False)]
    return "\n".join(filas)

def leer_salida_etl(lago, producto):
    """Lee el resultado del trabajo ETL (tablero_etl/<producto>/) como tabla de pandas."""
    t = pd.concat([pd.read_parquet(f) for f in sorted((lago / "tablero_etl" / producto).glob("*.parquet"))], ignore_index=True)
    t["fecha"] = t["fecha"].astype(str)
    return t

def correr_etl(lago, volumen, *args, **kw):
    """Ejecuta etl_tablero.py con la sesion ya abierta (no abre una segunda JVM) y devuelve su linea de bitacora."""
    base = ["--raiz", str(RAIZ), "--carpeta", str(lago), "--volumen", volumen, "--equipo", EQUIPO, "--bitacora", BITACORA]
    return T.ejecutar(spark, T.leer_parametros(base + list(args)))

## Nivel 1 · Guiado · 45 minutos

Se resuelve sobre la **base común** (200 medidores, 120 días, 576 000 lecturas), el día fijo **2024-02-15** y la ventana **2024-02-10 a 2024-02-15**. Las celdas ocuparán de 4 a 15 minutos de ejecución en conjunto, según la carga del equipo; el resto es lectura, contraste y decisión. Si el tiempo se acaba, difieran al Nivel 2 los sub-pasos D.3 y F.2, en ese orden.

### Sub-paso A · Contraste con la solución de referencia de la lectura de planes (4 minutos)

La celda imprime lo que el código de la capa de preparación declara de sí mismo y lo que dejó en disco. Con esa lectura y su registro de decisiones de la práctica anterior, completen la tabla de contraste de la guía: una fila por decisión (qué mantienen, qué cambiarían y por qué) con la pregunta nueva: **¿qué decisión de escritura hace barato o caro rehacer un día de la capa consolidada?** Una decisión distinta de la de referencia no es incorrecta si está bien justificada. Si su registro no está terminado, trabajen con decisiones provisionales y declárenlo.

In [3]:
# Sub-paso A: lo que la solucion de referencia de la capa de preparacion declara de si misma y lo que dejo en disco
fuente = (RAIZ / "pipeline" / "refinada_spark.py").read_text(encoding="utf-8").splitlines()
print("\n".join(fuente[1:36]))
archivos = sorted((LAGO_BASE / "refinada_spark").rglob("*.parquet"))
pf = pq.ParquetFile(archivos[0])
print()
print(f"archivos {len(archivos)} · {M.tamano_de(archivos):,} B · filas del primer archivo {pf.metadata.num_rows} · grupos de filas {pf.metadata.num_row_groups}")
print("columnas dentro del archivo:", pf.schema_arrow.names)
print("columnas que solo estan en la ruta: anio, mes, dia (texto con ceros)")


Este archivo es un ESQUELETO: definir_esquema, columnas_derivadas, clasificar_nativa y
escribir_capa las escriben ustedes (busquen TODO). La sesion de Spark, la lectura de los CSV, las
dos funciones propias (por fila y vectorizada), la firma por grupo, la reconciliacion y la
bitacora ya funcionan; el resto del texto de este encabezado describe el programa terminado.

Que hace
    Construye la capa refinada_spark/ del lago desde la capa CRUDA (CSV), no desde la refinada
    heredada, y la deja lista para que cualquier lector la entienda:
      1. Crea una sesion local de Spark (local[N]) con la zona horaria fijada en UTC.
      2. Lee los CSV con un ESQUEMA DECLARADO (tipos explicitos, ninguna columna inferida) y
         modo PERMISSIVE. Un tipo mal declarado deja nulos sin error: por eso se cuentan los
         nulos por columna antes de escribir.
      3. Deriva columnas: fecha, hora, es_confiable, nivel_presion (baja, normal o alta) y las
         tres partes de la ruta (anio, mes,

**Su decisión · Sub-paso A (equipo-1).**

| Decisión de la capa (T16 y T17) | ¿Mantener o cambiar? | Qué rehacer abarata o encarece |
|---|---|---|
| `anio`, `mes` y `dia` como texto con ceros | **Mantener.** Con texto, la vista limita la ventana con `concat(anio, mes, dia) BETWEEN …` y poda. | **Abarata** el reproceso por ventana: lee 6 archivos y no 120. |
| Un archivo por día | **Mantener.** | **Abarata** reprocesar un día: reescribe 1 archivo de unos 2 KB por producto y no un mes completo. |
| `fecha` dentro del archivo | **Mantener** como dato, pero **nunca** filtrar la ventana por `fecha`: en T17 abrió 120 archivos. | Filtrar por `fecha` **encarecería** el reproceso: abriría 120 archivos para rehacer 6 días. |
| La capa conserva todas las filas con `es_confiable` | **Mantener** las filas. **Cambiar** dónde se aplica la regla: el producto filtra por `quality_flag = 'OK'` en su SQL. | **Abarata** el cambio de regla: se rehace el producto (unos 0,9 MB de columnas en la base) y no se relee la cruda (29,6 MB). |


### Sub-paso B · Perfil `s18` y sesión (3 minutos)

Desde la terminal, **antes** de abrir el cuaderno: `docker compose --profile s12 down` y luego `docker compose --profile s18 up -d`. `s18` es **el mismo servicio** `spark` con una etiqueta de perfil más: los comandos se dirigen al servicio (`exec spark`), no al nombre del contenedor. La celda abre la sesión con la configuración fija (ejecución adaptativa apagada, 8 particiones de mezcla, sobrescritura dinámica) y muestra valores que conviene conocer: la **modalidad ANSI está apagada** (un `CAST` inválido da nulo sin error), el catálogo es **en memoria** (las tablas no sobreviven a la sesión) y el filtro `quality_flag = 'OK'` **se empuja al lector CSV**. Después registra la vista `prep_lecturas` sobre la preparación.

In [4]:
# Sub-paso B: sesion local[4] con la configuracion fija del analisis y la vista prep_lecturas sobre la preparacion
spark = E.abrir_sesion()
print("Spark", spark.version, "| master", spark.sparkContext.master)
for clave in ["spark.sql.ansi.enabled", "spark.sql.storeAssignmentPolicy", "spark.sql.catalogImplementation",
              "spark.sql.sources.partitionOverwriteMode", "spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions",
              "spark.sql.csv.filterPushdown.enabled"]:
    print(f"{clave:45s} {spark.conf.get(clave)}")
prep = E.registrar_prep(spark, LAGO_BASE)
print("vista prep_lecturas sobre refinada_spark/:", prep.count(), "filas")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 22:28:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 | master local[4]
spark.sql.ansi.enabled                        false
spark.sql.storeAssignmentPolicy               ANSI
spark.sql.catalogImplementation               in-memory
spark.sql.sources.partitionOverwriteMode      dynamic
spark.sql.adaptive.enabled                    false
spark.sql.shuffle.partitions                  8
spark.sql.csv.filterPushdown.enabled          true


vista prep_lecturas sobre refinada_spark/: 576000 filas


**Su decisión · Sub-paso B (decisión 7).** Fijamos UTC, la ejecución adaptativa apagada, 8 particiones de mezcla, columnas de ruta como texto, marca temporal en microsegundos y sobrescritura dinámica (`E.abrir_sesion`). Con esa configuración, el plan normalizado y los conteos son los mismos en cualquier equipo; solo cambia el tiempo. La regla vive en **SQL declarativo** (`sql/consolidada/*.sql`), y cada corrida registra el **hash del SQL** (`hash_sql`) y el **hash del plan normalizado** (en la ampliación, consumo `13adaf5020df` y presión `eb10a78640b1`, iguales en dos corridas): eso identifica qué versión de la regla produjo cada cifra.

### Sub-paso C · La línea base: el trabajo heredado del tablero (8 minutos)

`etl_tablero.py` es el trabajo que alimenta hoy el tablero: lee los CSV de la cruda con esquema declarado, **descarta** las lecturas cuya bandera no es `OK`, agrega por día y sector y guarda solo el resultado. Se escribió antes de que existiera la capa de preparación y lleva su propia copia de las reglas. **C.1** lo ejecuta, un trabajo por producto, y mide qué leyó y qué escribió; **C.2** hace la pregunta que su resultado no puede contestar.

In [5]:
# Sub-paso C.1: el trabajo heredado del tablero, un trabajo por producto (extrae la cruda, descarta, agrega y guarda solo el resultado)
for producto in ("consumo", "presion"):
    correr_etl(LAGO_BASE, "base", "--producto", producto)
    mostrar(ultima("base", accion="construir", enfoque="ETL", producto=producto))
b = ultima("base", accion="construir", enfoque="ETL", producto="consumo")
print(f"filas de salida del escaneo {b['filas_salida_escaneo']:,} (el filtro de calidad se empuja al lector CSV) · filas del resultado {b['filas_resultado']}")

ETL             consumo  archivos leidos  120 · bytes leidos  29,614,179 · escritos     6,667 B en 1 archivo(s) · etapas 2 · tareas 5 · mezcla 12,907 B


ETL             presion  archivos leidos  120 · bytes leidos  29,614,179 · escritos     5,608 B en 1 archivo(s) · etapas 2 · tareas 5 · mezcla 29,355 B
filas de salida del escaneo 541,368 (el filtro de calidad se empuja al lector CSV) · filas del resultado {'consumo': 480}


In [6]:
# Sub-paso C.2: la pregunta que un resultado que no retiene lo descartado no puede contestar
etl_consumo = leer_salida_etl(LAGO_BASE, "consumo")
fila = etl_consumo[(etl_consumo.fecha == DIA_FIJO) & (etl_consumo.sector == "S-ESTE")]
print("Resultado del ETL para S-ESTE el", DIA_FIJO)
print(fila.to_string(index=False))
cruda_dia = pd.read_csv(LAGO_BASE / "cruda" / "acueducto" / "anio=2024" / "mes=02" / "dia=15" / "lecturas.csv")
n_cruda = int((cruda_dia.sector == "S-ESTE").sum())
n_etl = int(fila.n_lecturas.iloc[0])
print(f"La cruda tiene {n_cruda} lecturas de ese sector y dia; el resultado cuenta {n_etl}. Las {n_cruda - n_etl} que faltan no estan en ningun lugar del resultado:")
print("para saber cuantas se descartaron hubo que volver a abrir un CSV de la cruda.")
assert n_cruda - n_etl > 0

Resultado del ETL para S-ESTE el 2024-02-15
     fecha sector  n_lecturas  consumo_total_m3
2024-02-15 S-ESTE        1136           182.246
La cruda tiene 1200 lecturas de ese sector y dia; el resultado cuenta 1136. Las 64 que faltan no estan en ningun lugar del resultado:
para saber cuantas se descartaron hubo que volver a abrir un CSV de la cruda.


**Su decisión · Sub-paso C (decisión 1).** El ETL leyó **los 120 CSV de la cruda (29 614 179 B)** para producir 480 filas, y su resultado **no permite saber cuántas lecturas descartó**: S-ESTE del 2024-02-15 tiene 1 136 en el resultado, frente a 1 200 en la cruda, y las 64 restantes no aparecen en ningún lado. El segundo producto (presión) le cuesta **otros 29 614 179 B**, porque el ETL vuelve a leer la cruda completa por producto.

### Sub-paso D · El producto ELT por SQL y sus reconciliaciones (10 minutos)

**D.1** construye `consumo_dia_sector` con SQL sobre `prep_lecturas`: cuenta **todas** las lecturas y separa las confiables de las no confiables; la regla se aplica al leer el producto. **D.2** hace las dos reconciliaciones (ELT filtrado frente a ETL; ELT sin filtro frente a `consolidada/`). **D.3** responde con una consulta la pregunta de C.2 y construye el segundo producto para comparar el costo marginal. En el plan, `ReadSchema` dice qué columnas lee el producto; la métrica declarada cuenta los bytes comprimidos de esas columnas.

In [7]:
# Sub-paso D.1: el producto ELT por SQL sobre la preparacion (la regla de calidad se aplica al leer el producto, no al construir la capa)
print((REF_SQL / "consumo_dia_sector.sql").read_text(encoding="utf-8"))
l = E.construir_y_registrar(spark, RAIZ, LAGO_BASE, "consumo", EQUIPO, "base", BITACORA, carpeta_sql=REF_SQL)
mostrar(l)
print("columnas que el plan lee:", l["columnas_leidas"])
print(f"filas de salida del escaneo {l['filas_salida_escaneo']:,} · filas del resultado {l['filas_resultado']} · hash del plan {l['hash_plan_normalizado'][:12]}")

-- Producto de consumo del tablero financiero: una fila por dia y sector.
--
-- Lee la vista prep_lecturas (la capa de preparacion refinada_spark/, ya limitada a la ventana de
-- dias por las columnas de la ruta). La preparacion CONSERVA todas las lecturas con su bandera de
-- calidad; la regla de confiabilidad se aplica AQUI, al construir el producto, y no antes:
--   n_lecturas            todas las lecturas del dia y sector (sin filtro)
--   n_confiables          las de quality_flag = 'OK'
--   n_no_confiables       el resto: queda contado, no desaparece
--   consumo_total_m3      suma de consumo de todas las lecturas
--   consumo_confiable_m3  suma de consumo de las confiables (lo que el trabajo heredado entrega)
-- anio, mes y dia van al final porque son las columnas de particion de la capa de salida.
SELECT
    fecha,
    sector,
    COUNT(*) AS n_lecturas,
    SUM(CASE WHEN quality_flag = 'OK' THEN 1 ELSE 0 END) AS n_confiables,
    SUM(CASE WHEN quality_flag = 'OK' THEN 0 ELSE 1

ELT             consumo  archivos leidos  120 · bytes leidos     860,254 · escritos   274,370 B en 120 archivo(s) · etapas 3 · tareas 20 · mezcla 53,232 B
columnas que el plan lee: ['sector', 'consumo_m3', 'quality_flag', 'fecha']
filas de salida del escaneo 576,000 · filas del resultado 480 · hash del plan 97537744a657


In [8]:
# Sub-paso D.2: dos reconciliaciones. (1) El ELT filtrado debe dar lo mismo que el ETL. (2) El ELT sin filtro debe coincidir con consolidada/.
elt = E.leer_producto_pandas(LAGO_BASE / "consolidada_elt" / "consumo_dia_sector")
etl_consumo = leer_salida_etl(LAGO_BASE, "consumo")
m = etl_consumo.merge(elt, on=["fecha", "sector"], suffixes=("_etl", "_elt"))
dist = int((m.n_lecturas_etl != m.n_confiables).sum())
dif = float((m.consumo_total_m3_etl - m.consumo_confiable_m3).abs().max())
print(f"(1) ELT filtrado frente a ETL: {len(m)} grupos · conteos distintos {dist} · diferencia maxima de consumo {dif:.1e}")
ref = R.referencia_consolidada(LAGO_BASE)
m2 = elt.merge(ref, on=["fecha", "sector"], suffixes=("", "_ref"))
dist2 = int((m2.n_lecturas != m2.n_lecturas_ref).sum())
dif2 = float((m2.consumo_total_m3 - m2.consumo_total_m3_ref).abs().max())
print(f"(2) ELT sin filtro frente a consolidada/: {len(m2)} grupos · conteos distintos {dist2} · diferencia maxima de consumo {dif2:.1e}")
tot = pd.DataFrame({"lecturas": [int(elt.n_lecturas.sum()), int(elt.n_confiables.sum()), int(elt.n_no_confiables.sum())],
                    "consumo_m3": [round(float(elt.consumo_total_m3.sum()), 3), round(float(elt.consumo_confiable_m3.sum()), 3), round(float(elt.consumo_total_m3.sum() - elt.consumo_confiable_m3.sum()), 3)]},
                   index=["guardadas (todas)", "confiables (las que cuenta el tablero)", "no confiables (retenidas con bandera)"])
print(tot.to_string())
assert len(m) == 480 and dist == 0 and dif <= 1e-6 and len(m2) == 480 and dist2 == 0 and dif2 <= 1e-6

(1) ELT filtrado frente a ETL: 480 grupos · conteos distintos 0 · diferencia maxima de consumo 0.0e+00


(2) ELT sin filtro frente a consolidada/: 480 grupos · conteos distintos 0 · diferencia maxima de consumo 5.7e-13
                                        lecturas  consumo_m3
guardadas (todas)                         576000   92166.022
confiables (las que cuenta el tablero)    541368   86607.816
no confiables (retenidas con bandera)      34632    5558.206


In [9]:
# Sub-paso D.3: la pregunta de C.2 respondida con una consulta, y el segundo producto (presion por nivel)
elt = E.leer_producto_pandas(LAGO_BASE / "consolidada_elt" / "consumo_dia_sector")
print(elt[(elt.fecha == DIA_FIJO) & (elt.sector == "S-ESTE")][["fecha", "sector", "n_lecturas", "n_confiables", "n_no_confiables", "consumo_total_m3", "consumo_confiable_m3"]].to_string(index=False))
lp = E.construir_y_registrar(spark, RAIZ, LAGO_BASE, "presion", EQUIPO, "base", BITACORA, carpeta_sql=REF_SQL)
mostrar(lp)
etl_p = ultima("base", accion="construir", enfoque="ETL", producto="presion")
tabla = pd.DataFrame({"enfoque": ["ETL (relee la cruda)", "ELT (lee la preparacion)"],
                      "bytes leidos por el segundo producto": [etl_p["bytes_leidos"], lp["bytes_leidos"]],
                      "bytes escritos": [etl_p["bytes_escritos"], lp["bytes_escritos"]]})
print(tabla.to_string(index=False))

     fecha sector  n_lecturas  n_confiables  n_no_confiables  consumo_total_m3  consumo_confiable_m3
2024-02-15 S-ESTE        1200          1136               64           192.831               182.246


ELT             presion  archivos leidos  120 · bytes leidos     236,202 · escritos   255,328 B en 120 archivo(s) · etapas 3 · tareas 20 · mezcla 110,881 B
                 enfoque  bytes leidos por el segundo producto  bytes escritos
    ETL (relee la cruda)                              29614179            5608
ELT (lee la preparacion)                                236202          255328


**Su decisión · Sub-paso D (decisión 2).** La **regla de confiabilidad** vive en el **producto**: el SQL aplica `quality_flag = 'OK'` al calcular `n_confiables` y `consumo_confiable_m3`, y la preparación conserva las 576 000 lecturas. Si la regla cambia, se reescribe el SQL y se rehace el producto: en la base, 860 254 B de columnas frente a 29 614 179 B del ETL. Los **umbrales de presión** viven en la **preparación**, con una sola copia (`nivel_presion`, en `refinada_spark.py`). Si cambian, hay que reconstruir la preparación, que es más caro, pero cambian mucho menos que la regla de confiabilidad. El ETL heredado tenía su propia copia de los umbrales, y dos copias pueden divergir.

### Sub-paso E · Cambio de regla y reproceso (8 minutos)

**E.1** cambia la regla ("`ESTIMADO` también cuenta") y mide qué se relee en cada enfoque: el ETL vuelve a la cruda; el ELT relee columnas de la preparación. **E.2** reprocesa la ventana de 6 días sobre una **copia** de la capa de salida con sobrescritura **dinámica** (dos corridas, para ver que es idempotente) y **estática**, y compara el árbol de archivos antes y después.

In [10]:
# Sub-paso E.1: la regla de confiabilidad cambia ("ESTIMADO" tambien cuenta). Que se relee y cuanto en cada enfoque.
REG = RAIZ / "_regla"
shutil.rmtree(REG, ignore_errors=True)
(REG / "sql").mkdir(parents=True)
nuevo = (REF_SQL / "consumo_dia_sector.sql").read_text(encoding="utf-8").replace("quality_flag = 'OK'", "quality_flag IN ('OK', 'ESTIMADO')")
(REG / "sql" / "consumo_dia_sector.sql").write_text(nuevo, encoding="utf-8")
# ETL: la regla vive en el codigo del trabajo y hay que volver a ejecutarlo contra la cruda
b_etl = correr_etl(LAGO_BASE, "base", "--producto", "consumo", "--estados-confiables", "OK,ESTIMADO", "--salida", str(REG / "tablero_etl"), "--bitacora", "_regla/bitacora.jsonl")
# ELT: la regla vive en el SQL del producto y se relee la preparacion, no la cruda
E.registrar_prep(spark, LAGO_BASE)
plan, _ = E.plan_producto(spark, REG / "sql" / "consumo_dia_sector.sql")
E.construir_producto(spark, REG / "sql" / "consumo_dia_sector.sql", REG / "consolidada_elt")
nuevo_elt = E.leer_producto_pandas(REG / "consolidada_elt")
bytes_elt = M.bytes_leidos("parquet", E.archivos_seleccionados(LAGO_BASE / "refinada_spark"), plan)
etl_n = pd.concat([pd.read_parquet(f) for f in (REG / "tablero_etl" / "consumo").glob("*.parquet")])
print(f"lecturas confiables con la regla nueva: ELT {int(nuevo_elt.n_confiables.sum()):,} · ETL {int(etl_n.n_lecturas.sum()):,} (antes 541 368)")
print(f"lecturas no confiables ahora: {int(nuevo_elt.n_no_confiables.sum()):,}")
print(f"bytes leidos para rehacer el producto: ETL {b_etl['bytes_leidos']:,} (la cruda) · ELT {bytes_elt:,} (columnas de la preparacion)")
assert int(nuevo_elt.n_confiables.sum()) == int(etl_n.n_lecturas.sum()) == 570181

lecturas confiables con la regla nueva: ELT 570,181 · ETL 570,181 (antes 541 368)
lecturas no confiables ahora: 5,819
bytes leidos para rehacer el producto: ETL 29,614,179 (la cruda) · ELT 860,254 (columnas de la preparacion)


In [11]:
# Sub-paso E.2: reprocesar una ventana de 6 dias sobre una COPIA de la capa de salida: dos corridas dinamicas y una estatica
tabla = "consumo_dia_sector"
origen, copia = LAGO_BASE / "consolidada_elt" / tabla, RAIZ / "_copias_reproceso" / tabla
corridas = []
for modo in ("dinamica", "dinamica", "estatica"):
    E.copiar_salida(origen, copia)
    corridas.append(E.reprocesar_y_registrar(spark, RAIZ, LAGO_BASE, "consumo", VENTANA_N1, EQUIPO, "base", copia, modo, BITACORA, carpeta_sql=REF_SQL))
resumen = pd.DataFrame([{"sobrescritura": c["sobrescritura"], "intactas": c["particiones_intactas"], "reescritas": c["particiones_reescritas"],
                         "perdidas": c["particiones_perdidas"], "archivos leidos": c["archivos_leidos"], "bytes de archivos": c["bytes_archivos_seleccionados"],
                         "bytes de columnas": c["bytes_leidos"], "bytes escritos": c["bytes_escritos"], "hash": c["hash_resultado"][:10]} for c in corridas])
print(resumen.to_string(index=False))
assert corridas[0]["hash_resultado"] == corridas[1]["hash_resultado"], "dos corridas dinamicas deben dar el mismo hash"
assert (corridas[0]["particiones_intactas"], corridas[0]["particiones_perdidas"]) == (114, 0) and corridas[2]["particiones_perdidas"] == 114

sobrescritura  intactas  reescritas  perdidas  archivos leidos  bytes de archivos  bytes de columnas  bytes escritos       hash
     dinamica       114           6         0                6             150774              43159           13685 9cf8f04d6a
     dinamica       114           6         0                6             150774              43159           13685 9cf8f04d6a
     estatica         0           6       114                6             150774              43159           13685 3a22099ec4


**Su decisión · Sub-paso E (decisión 4).** Elegimos la **sobrescritura dinámica**. Sobre la copia de la base, las dos corridas dinámicas de la ventana dejaron **114 particiones intactas, 6 reescritas y 0 perdidas**, con el **mismo hash** (`9cf8f04d6a`): el reproceso es idempotente y solo tocó la ventana. La estática, con la misma lectura (6 archivos), perdió **114 particiones**.

### Sub-paso F · Contratos entre capas (7 minutos)

**F.1** verifica los contratos de las dos fronteras (cruda a preparación y preparación a consolidada) con los datos del proyecto: esquema, grano y reconciliaciones, con pandas y pyarrow, sin abrir Spark. **F.2** evalúa la **copia alterada** que se preparó antes de la clase: un día de la cruda con la coma decimal entrecomillada (`"0,209"`). El archivo sigue siendo un CSV válido, la lectura permisiva convierte lo ilegible en nulo, el ETL entrega las mismas 480 filas con menos consumo y **sin ningún aviso**; el contrato falla en la frontera correcta.

In [12]:
# Sub-paso F.1: los contratos de las dos fronteras, aprobados con los datos del proyecto
prep_dir, cruda_dir = LAGO_BASE / "refinada_spark", LAGO_BASE / "cruda" / "acueducto"
casos = [(REF_CONTRATOS / "preparacion_a_consolidada.json", E.tablas_preparacion_a_consolidada(prep_dir, LAGO_BASE / "consolidada_elt" / "consumo_dia_sector")),
         (REF_CONTRATOS / "cruda_a_preparacion.json", E.tablas_cruda_a_preparacion(prep_dir, cruda_dir))]
for ruta, datos in casos:
    res, linea = E.evaluar_y_registrar_contrato(RAIZ, ruta, datos, "refinada_spark", EQUIPO, "base", BITACORA)
    print(f"frontera {linea['frontera']}: {'APROBADO' if linea['aprobado'] else 'NO APROBADO'} ({sum(r['ok'] for r in res)} de {len(res)} invariantes)")
    for r in res:
        print(f"   {r['invariante']} {r['tipo']:20s} {'ok' if r['ok'] else 'FALLA'}")
    assert linea["aprobado"]

frontera preparacion_a_consolidada: APROBADO (5 de 5 invariantes)
   PC1 conteo_igual         ok
   PC2 unico                ok
   PC3 compuesto            ok
   PC4 sin_nulos            ok
   PC5 suma_igual           ok
frontera cruda_a_preparacion: APROBADO (6 de 6 invariantes)
   C1 columnas_requeridas  ok
   C2 valores_en           ok
   C3 sin_nulos            ok
   C4 conteo_igual         ok
   C5 suma_igual           ok
   C6 igual_por_grupo      ok


In [13]:
# Sub-paso F.2: la copia alterada (un dia con la coma decimal entrecomillada). El ETL no avisa; el contrato falla en la frontera correcta.
DERIVA = RAIZ / "_deriva"
assert (DERIVA / "refinada_spark").exists(), 'Falta la preparacion de la copia alterada: ejecuten los comandos 4 y 5 de "Antes de empezar" con el cuaderno cerrado'
correr_etl(DERIVA, "base", "--producto", "consumo", "--salida", str(DERIVA / "tablero_etl"), "--bitacora", "_deriva/bitacora_capas.jsonl")
real, copia = leer_salida_etl(LAGO_BASE, "consumo"), leer_salida_etl(DERIVA, "consumo")
print(f"ETL sobre la copia, sin error: {len(copia)} filas (antes {len(real)}) · {int(copia.n_lecturas.sum()):,} lecturas (antes {int(real.n_lecturas.sum()):,})")
print(f"consumo total del resultado: {copia.consumo_total_m3.sum():,.3f} m3 frente a {real.consumo_total_m3.sum():,.3f} m3 -> faltan {real.consumo_total_m3.sum() - copia.consumo_total_m3.sum():,.3f} m3")
res, linea = E.evaluar_y_registrar_contrato(RAIZ, REF_CONTRATOS / "cruda_a_preparacion.json",
                                            E.tablas_cruda_a_preparacion(DERIVA / "refinada_spark", LAGO_BASE / "cruda" / "acueducto"),
                                            "_deriva/refinada_spark", EQUIPO, "base", BITACORA)
for r in res:
    print(f"   {r['invariante']} {r['tipo']:20s} {'ok' if r['ok'] else 'FALLA'}  observado {str(r['observado'])[:70]}")
print("primer invariante que falla:", linea["invariante_que_falla"])
assert not linea["aprobado"] and len(copia) == len(real) and linea["invariante_que_falla"] == "C3"

ETL sobre la copia, sin error: 480 filas (antes 480) · 541,368 lecturas (antes 541,368)
consumo total del resultado: 85,879.692 m3 frente a 86,607.816 m3 -> faltan 728.124 m3


   C1 columnas_requeridas  ok  observado ['meter_id', 'timestamp', 'sector', 'consumo_m3', 'presion_bar', 'qual
   C2 valores_en           ok  observado []
   C3 sin_nulos            FALLA  observado {'meter_id': 0, 'timestamp': 0, 'consumo_m3': 4800}
   C4 conteo_igual         ok  observado 576000
   C5 suma_igual           FALLA  observado 91394.173
   C6 igual_por_grupo      FALLA  observado {'grupos_sin_par': 0, 'grupos_distintos': 4, 'dif_max': 193.8}
primer invariante que falla: C3


**Su decisión · Sub-paso F (decisión 6).** En nuestro contrato `cruda_a_preparacion`, el día con el consumo ilegible lo detectan **C2 `sin_nulos`** (miles de `consumo_m3` nulos, el primero que falla), **C5 `suma_igual`** (el consumo de la preparación no es el de la cruda) y **C6 `igual_por_grupo`** (4 grupos de ese día con distinta suma). Se verifica **en esa frontera** porque es donde el dato se daña: un valor ilegible se vuelve nulo al tipar la columna. En el tablero ya no hay nada que lo delate: el ETL sobre la copia terminó sin error, con las mismas filas y menos consumo.

### Cierre del Nivel 1

La celda reúne las mediciones en `comparacion_nivel1.csv`, registra la prueba de que el lago heredado no cambió y la tabla comparativa de la guía se completa con una frase por compromiso (costo, trazabilidad, reproceso, contratos).

In [14]:
# Cierre del Nivel 1: tabla comparativa, prueba de que el lago heredado no cambio y comparacion_nivel1.csv
tabla = E.escribir_comparacion(RAIZ, LAGO_BASE, "base", RAIZ / "comparacion_nivel1.csv", BITACORA)
print(tabla[["producto", "enfoque", "archivos_leidos", "bytes_leidos", "bytes_escritos", "filas_descartadas_visibles"]].to_string(index=False))
despues = {"lago": V.hash_arbol_lago(LAGO_BASE), "preparacion": V.hash_arbol_capa(LAGO_BASE / "refinada_spark")}
E.registrar_integridad(RAIZ, LAGO_BASE, "base", HASHES_BASE_ANTES, despues, EQUIPO, BITACORA)
print("lago heredado (datalake/) sin cambios:", despues == HASHES_BASE_ANTES)
assert despues == HASHES_BASE_ANTES

producto enfoque  archivos_leidos  bytes_leidos  bytes_escritos  filas_descartadas_visibles
 consumo     ELT              120        860254          274370                       34632
 consumo     ETL              120      29614179            6667                           0
 presion     ELT              120        236202          255328                       34632
 presion     ETL              120      29614179            5608                           0


lago heredado (datalake/) sin cambios: True


**Consolidación (por equipo).**

(1) Decidiríamos con los **bytes leídos por producto**. En la ampliación, el ELT lee 1 382 744 B (consumo) y 401 505 B (presión), y el ETL lee 54 371 830 B por producto. La decisión cambiaría si la preparación **no existiera** y hubiera que construirla ($K$ = 113 MB con dos lecturas de la cruda): con 2 productos fijos y reglas estables, el ETL de una pasada sería más barato.

(2) Se reprocesa **solo la capa consolidada** (los productos), no la preparación. Se leen las columnas del producto: en la ampliación, 1 382 744 B para consumo, o 69 278 B si el cambio afecta solo una ventana de 6 días.

(3) Para la dirección financiera: *"El tablero muestra el consumo de las lecturas confiables; guardamos además las que no lo son, marcadas, para poder explicar la diferencia y recalcular si la regla cambia."*

## Nivel 2 · Aplicado · 75 minutos dentro de las 3 horas de la tarea

**Presupuesto (estimado).** Comprobar la ampliación y escribir la ventana, 10 min; hipótesis, 5 min; SQL de los dos productos y su reconciliación, 15 min; contratos y copia alterada, 15 min; reproceso e idempotencia, 10 min; mediciones de los cuatro modos del ETL frente al ELT y bitácora, 12 min; verificación y evidencia, 5 min; colchón, 3 min. La ejecución ocupará de 7 a 20 minutos según la carga del equipo (el ELT escribe 120 particiones por producto y por corrida); lo que consume el tiempo es escribir y redactar.

> **Antes del Nivel 2.** La ampliación de su equipo, su capa de preparación y su referencia ya deben existir (los tres comandos de "Antes de empezar" de la guía, con este cuaderno cerrado). La celda N2.1 lo comprueba.

### El encargo

Repitan el patrón del Nivel 1 sobre **la ampliación de su equipo**, con cuatro piezas **que ustedes escriben**: la ventana de reproceso, los dos SQL, los dos contratos y las tres hipótesis. Sus cifras serán distintas de las de otro equipo (otro volumen, otra ventana); la estructura de los resultados, no. Cambien `EQUIPO` en la celda de configuración (o definan la variable de entorno `EQUIPO`) y ejecuten en orden.

In [15]:
# N2.1 Comprobar la ampliacion del equipo y fijar el volumen del Nivel 2
LAGO2, VOL2 = LAGO_EQUIPO, "ampliacion"
for capa in ("cruda", "refinada", "refinada_spark", "referencia"):
    assert (LAGO2 / capa).exists(), f"Falta {LAGO2.name}/{capa}: ejecuten los tres comandos de 'Antes de empezar' con el cuaderno cerrado"
totales = pd.read_csv(LAGO2 / "totales_control.csv") if (LAGO2 / "totales_control.csv").exists() else R.referencia_consolidada(LAGO2)
archivos2 = sorted((LAGO2 / "refinada_spark").rglob("*.parquet"))
print(f"{LAGO2.name}: {int(totales.n_lecturas.sum()):,} lecturas · {len(archivos2)} archivos de preparacion · {M.tamano_de(archivos2):,} B")
HASHES2_ANTES = {"lago": V.hash_arbol_lago(LAGO2), "preparacion": V.hash_arbol_capa(LAGO2 / "refinada_spark")}
E.registrar_prep(spark, LAGO2)

datalake_equipo: 1,002,240 lecturas · 120 archivos de preparacion · 4,519,275 B


DataFrame[meter_id: string, timestamp: timestamp, sector: string, consumo_m3: double, presion_bar: double, quality_flag: string, fecha: date, hora: int, es_confiable: boolean, nivel_presion: string, anio: string, mes: string, dia: string]

### N2.2 Su turno (1/4): la ventana de reproceso

**Decisión de ingeniería de este nivel:** la ventana define la unidad de reproceso. Es la regla `dia_de_a` de la práctica anterior (el día de mayor consumo del equipo) con 6 días que terminan en él.

In [16]:
# N2.2 (1/4) Su turno: la ventana de reproceso del equipo
import datetime as dt
import planes_s17 as P          # P.dia_de_a(totales) devuelve el dia de mayor consumo como texto 'AAAA-MM-DD'

def ventana_de_equipo(totales_control, dias=6):
    """Ventana de reproceso del equipo: los `dias` dias que terminan en el dia de mayor consumo del equipo
    (P.dia_de_a, el mismo dia de la consulta A de la lectura de planes), recortados al primer dia de la historia
    (el menor valor de la columna fecha de totales_control). Devuelve (desde, hasta) como texto 'AAAA-MM-DD'."""
    hasta = dt.date.fromisoformat(P.dia_de_a(totales_control))
    primero = dt.date.fromisoformat(str(totales_control["fecha"].min()))
    desde = max(primero, hasta - dt.timedelta(days=dias - 1))
    return desde.isoformat(), hasta.isoformat()


In [17]:
# Autoverificacion de la ventana (si falla, revisen la funcion antes de seguir)
import datetime as dt
desde, hasta = ventana_de_equipo(totales)
dias_ventana = (dt.date.fromisoformat(hasta) - dt.date.fromisoformat(desde)).days + 1
assert hasta == E.P.dia_de_a(totales), "la ventana debe terminar en el dia de mayor consumo del equipo"
assert desde >= str(totales["fecha"].min()) and 1 <= dias_ventana <= 6
VENTANA2 = (desde, hasta)
print(f"ventana del equipo: {desde} a {hasta} ({dias_ventana} dias)")

ventana del equipo: 2024-01-09 a 2024-01-14 (6 dias)


**N2.2 (2/4) Llevar la función al módulo.** Abran `pipeline/elt_s18.py`, **reemplacen la función entera** `ventana_de_equipo` (la firma, el docstring y el cuerpo, sin dejar su `NotImplementedError` ni la palabra `TODO`) por la suya y guarden. La celda siguiente recarga el módulo y detecta cualquier marca pendiente.

In [18]:
# N2.2 (2/4) Llevar la funcion al modulo: reemplacen la funcion ENTERA (docstring incluida) en pipeline/elt_s18.py y recarguen el modulo
importlib.reload(E)
pendientes = E.sin_pendientes([RAIZ / "pipeline" / "elt_s18.py"])
assert not pendientes, f"Faltan marcas por quitar en elt_s18.py (NotImplementedError o TODO): {pendientes}"
assert E.ventana_de_equipo(totales) == VENTANA2
print("pipeline/elt_s18.py sin NotImplementedError ni TODO; la ventana del modulo coincide con la de la celda")

pipeline/elt_s18.py sin NotImplementedError ni TODO; la ventana del modulo coincide con la de la celda


### N2.3 Su turno (3/4): hipótesis antes de medir

Escriban un valor numérico por hipótesis **antes** de ejecutar el ELT. La bitácora guarda su marca de tiempo; la etapa `t18` comprueba que es anterior a la medición.

In [19]:
# N2.3 Su turno: tres hipotesis numericas, ANTES de medir
# equipo-1: prediccion hecha con los metadatos de Parquet de la preparacion (bytes comprimidos por columna, leidos con
# pyarrow sobre los 120 archivos), ANTES de ejecutar el ELT. Consumo lee fecha, sector, consumo_m3 y quality_flag
# (7 672 + 49 200 + 1 178 255 + 147 617 B, unos 1,38 MB); presion lee fecha, sector, nivel_presion y quality_flag
# (7 672 + 49 200 + 197 016 + 147 617 B, unos 0,40 MB). Se redondea porque la metrica del modulo puede contar
# encabezados o paginas de diccionario de otra forma. El reproceso reescribe los 6 dias de la ventana.
hipotesis = {
    "elt_consumo": {"clave": "bytes_columnas_producto1", "texto": "Bytes de columnas que leera el producto de consumo ELT (fecha, sector, consumo_m3, quality_flag; suma de bytes comprimidos por columna de los 120 archivos)", "valor": 1_380_000},
    "elt_presion": {"clave": "bytes_columnas_producto2", "texto": "Bytes de columnas que leera el producto de presion ELT (fecha, sector, nivel_presion, quality_flag; sin consumo_m3, la columna mas pesada despues de presion_bar)", "valor": 400_000},
    "reproceso": {"clave": "particiones_reescritas", "texto": "Particiones que reescribira el reproceso de la ventana de 6 dias con sobrescritura dinamica (una por dia)", "valor": 6},
}
E.registrar_hipotesis(RAIZ / "hipotesis_s18.json", hipotesis)


{'elt_consumo': {'clave': 'bytes_columnas_producto1',
  'texto': 'Bytes de columnas que leera el producto de consumo ELT (fecha, sector, consumo_m3, quality_flag; suma de bytes comprimidos por columna de los 120 archivos)',
  'valor': 1380000,
  'marca_hipotesis': '2026-11-12T22:28:36.648051Z'},
 'elt_presion': {'clave': 'bytes_columnas_producto2',
  'texto': 'Bytes de columnas que leera el producto de presion ELT (fecha, sector, nivel_presion, quality_flag; sin consumo_m3, la columna mas pesada despues de presion_bar)',
  'valor': 400000,
  'marca_hipotesis': '2026-11-12T22:28:36.648058Z'},
 'reproceso': {'clave': 'particiones_reescritas',
  'texto': 'Particiones que reescribira el reproceso de la ventana de 6 dias con sobrescritura dinamica (una por dia)',
  'valor': 6,
  'marca_hipotesis': '2026-11-12T22:28:36.648060Z'}}

In [20]:
# Autoverificacion: hipotesis numericas, con marca de tiempo y dentro de un rango plausible
guardadas = json.loads((RAIZ / "hipotesis_s18.json").read_text(encoding="utf-8"))
assert set(guardadas) == {"elt_consumo", "elt_presion", "reproceso"}
assert all(isinstance(h["valor"], (int, float)) and h.get("marca_hipotesis") for h in guardadas.values())
tamano_prep = M.tamano_de(archivos2)
for k in ("elt_consumo", "elt_presion"):
    assert 1_000 < guardadas[k]["valor"] < tamano_prep, f"{k}: un producto lee solo cuatro de las diez columnas del archivo, es decir menos que toda la preparacion ({tamano_prep:,} B)"
assert 1 <= guardadas["reproceso"]["valor"] <= 6, "el reproceso reescribe entre 1 y 6 particiones (los dias de la ventana)"
print("hipotesis registradas, cada una con su marca de tiempo:", sorted(guardadas))

hipotesis registradas, cada una con su marca de tiempo: ['elt_consumo', 'elt_presion', 'reproceso']


### N2.4 Su turno (4/4): los dos productos en SQL

**Decisiones de ingeniería de este nivel:** qué columnas lee cada producto (cada columna extra cuesta bytes) y que las columnas de partición (`anio`, `mes`, `dia`) vayan al final del `SELECT`, en ese orden.

In [21]:
# N2.4 Su turno: los dos productos ELT en SQL sobre prep_lecturas
# Cada uno agrupa por anio, mes, dia (columnas de particion de la capa), fecha y sector (y nivel_presion en el segundo).
SQL_CONSUMO = """
-- Producto consumo_dia_sector (equipo-1): consumo por dia y sector sobre la vista prep_lecturas.
-- Cuenta TODAS las lecturas de la preparacion y separa confiables de no confiables en lugar de descartarlas.
-- La regla de confiabilidad (quality_flag = 'OK') se aplica AQUI, al leer el producto: cambiarla es cambiar este
-- SQL y reprocesar el producto, sin reconstruir la preparacion. Las columnas de particion van al final.
SELECT
    fecha,
    sector,
    COUNT(*)                                            AS n_lecturas,
    SUM(CASE WHEN quality_flag = 'OK' THEN 1 ELSE 0 END)  AS n_confiables,
    SUM(CASE WHEN quality_flag = 'OK' THEN 0 ELSE 1 END)  AS n_no_confiables,
    SUM(consumo_m3)                                     AS consumo_total_m3,
    SUM(CASE WHEN quality_flag = 'OK' THEN consumo_m3 END) AS consumo_confiable_m3,
    anio,
    mes,
    dia
FROM prep_lecturas
GROUP BY fecha, sector, anio, mes, dia
"""
SQL_PRESION = """
-- Producto presion_dia_sector (equipo-1): lecturas por dia, sector y nivel de presion sobre prep_lecturas.
-- Usa la clase nivel_presion que ya calculo la preparacion (una sola copia de los umbrales 2,5 y 3,5 bar);
-- una presion nula queda como 'sin_dato', la misma etiqueta del tablero heredado. Cuenta todas las lecturas
-- y separa confiables (quality_flag = 'OK') de no confiables. Lee cuatro columnas de la preparacion (mas las de la ruta).
SELECT
    fecha,
    sector,
    COALESCE(nivel_presion, 'sin_dato')                 AS nivel_presion,
    COUNT(*)                                            AS n_lecturas,
    SUM(CASE WHEN quality_flag = 'OK' THEN 1 ELSE 0 END)  AS n_confiables,
    SUM(CASE WHEN quality_flag = 'OK' THEN 0 ELSE 1 END)  AS n_no_confiables,
    anio,
    mes,
    dia
FROM prep_lecturas
GROUP BY fecha, sector, COALESCE(nivel_presion, 'sin_dato'), anio, mes, dia
"""
(RAIZ / "sql" / "consolidada" / "consumo_dia_sector.sql").write_text(SQL_CONSUMO.lstrip("\n"), encoding="utf-8")
(RAIZ / "sql" / "consolidada" / "presion_dia_sector.sql").write_text(SQL_PRESION.lstrip("\n"), encoding="utf-8")
print("SQL escritos en sql/consolidada/")


SQL escritos en sql/consolidada/


In [22]:
# Autoverificacion de los SQL: sin marcas pendientes, con la bandera aplicada y las columnas de particion al final
for nombre in ("consumo_dia_sector", "presion_dia_sector"):
    texto = (RAIZ / "sql" / "consolidada" / f"{nombre}.sql").read_text(encoding="utf-8")
    assert "TODO" not in texto and "prep_lecturas" in texto and "quality_flag" in texto
    assert re.search(r"anio,\s*mes,\s*dia\s+FROM", texto), "anio, mes y dia deben ser las ultimas columnas del SELECT"
print("los dos SQL tienen la forma esperada")

los dos SQL tienen la forma esperada


### N2.5 Su turno: los contratos

**Decisiones de ingeniería de este nivel:** qué invariantes y en qué frontera (decisión 6). Cada contrato necesita esquema, grano y al menos tres invariantes, uno de reconciliación.

In [23]:
# N2.5 Su turno: completar los dos contratos (esquema, grano e invariantes de reconciliacion)
# Tipos de invariante: columnas_requeridas, sin_nulos, unico, valores_en, conteo_igual, suma_igual, compuesto, igual_por_grupo
# equipo-1: los invariantes se escribieron directamente en contratos/*.json (versionados); esta celda los muestra.
cp = json.loads((RAIZ / "contratos" / "preparacion_a_consolidada.json").read_text(encoding="utf-8"))
cc = json.loads((RAIZ / "contratos" / "cruda_a_preparacion.json").read_text(encoding="utf-8"))
for c in (cc, cp):
    print(c["frontera"], "· grano", c["grano"])
    for i in c["invariantes"]:
        print(f"   {i['id']:4s} {i['tipo']:18s} {i['descripcion']}")


cruda_a_preparacion · grano ['meter_id', 'timestamp']
   C1   columnas_requeridas Las seis columnas de la cruda aparecen primero y en el mismo orden
   C2   sin_nulos          Ninguna lectura ilegible se convirtio en nulo: las columnas no nulables no tienen nulos
   C3   unico              Grano: una fila por (meter_id, timestamp)
   C4   conteo_igual       La preparacion tiene exactamente las filas que llegaron a la cruda
   C5   suma_igual         El consumo total de la preparacion es el de la cruda
   C6   igual_por_grupo    Mismas filas y mismo consumo por (fecha, sector) que la cruda
   C7   valores_en         Solo los cuatro sectores del acueducto
preparacion_a_consolidada · grano ['fecha', 'sector']
   PC1  conteo_igual       La suma de n_lecturas es el numero de filas de la preparacion
   PC2  unico              Grano: una fila por (fecha, sector)
   PC3  sin_nulos          Las columnas no nulables del producto no tienen nulos
   PC4  compuesto          Confiables mas no confia

In [24]:
# Autoverificacion de los contratos: sin marcas pendientes, con esquema, grano y al menos tres invariantes (uno de reconciliacion)
for nombre in ("preparacion_a_consolidada", "cruda_a_preparacion"):
    c = json.loads((RAIZ / "contratos" / f"{nombre}.json").read_text(encoding="utf-8"))
    assert "pendiente" not in c and "TODO" not in json.dumps(c)
    assert c["columnas"] and c["grano"] and len(c["invariantes"]) >= 3
    assert any(i["tipo"] in ("conteo_igual", "suma_igual") for i in c["invariantes"]), "falta un invariante de reconciliacion"
print("los dos contratos tienen la forma esperada")

los dos contratos tienen la forma esperada


### N2.6 Mediciones: cuatro modos del ETL, ELT y resultado idéntico

Las corridas del ETL se proveen (un trabajo por producto, una pasada, ventana por ruta y el control de formato que lee la misma preparación en Parquet). Después el ELT con sus SQL y su hipótesis. **Decisión de ingeniería de este nivel:** qué producto va por ETL y cuál por ELT (decisión 1) y qué métrica de bytes se usa.

In [25]:
# N2.6 Los cuatro modos del ETL y el control de formato sobre su ampliacion (corridas provistas)
c = lambda *a: correr_etl(LAGO2, VOL2, *a)
c("--producto", "consumo"); c("--producto", "presion")                       # ETL: un trabajo por producto
c("--producto", "ambos")                                                     # ETL-una-pasada: lee la cruda una vez
c("--producto", "consumo", "--modo", "ventana", "--desde", VENTANA2[0], "--hasta", VENTANA2[1])   # ETL-ventana: solo las carpetas de la ventana
c("--producto", "consumo", "--formato", "parquet"); c("--producto", "presion", "--formato", "parquet")   # control de formato
for e, p in (("ETL", "consumo"), ("ETL", "presion"), ("ETL-una-pasada", "ambos"), ("ETL-ventana", "consumo"), ("ETL-parquet", "consumo"), ("ETL-parquet", "presion")):
    mostrar(ultima(VOL2, accion="construir", enfoque=e, producto=p))

ETL             consumo  archivos leidos  120 · bytes leidos  54,371,830 · escritos     6,852 B en 1 archivo(s) · etapas 2 · tareas 6 · mezcla 14,059 B
ETL             presion  archivos leidos  120 · bytes leidos  54,371,830 · escritos     6,238 B en 1 archivo(s) · etapas 2 · tareas 6 · mezcla 31,094 B
ETL-una-pasada  ambos    archivos leidos  120 · bytes leidos  54,371,830 · escritos    12,996 B en 2 archivo(s) · etapas 4 · tareas 15 · mezcla 77,546 B
ETL-ventana     consumo  archivos leidos    6 · bytes leidos   2,718,973 · escritos     8,150 B en 6 archivo(s) · etapas 3 · tareas 19 · mezcla 4,018 B
ETL-parquet     consumo  archivos leidos  120 · bytes leidos   1,382,744 · escritos     7,007 B en 1 archivo(s) · etapas 2 · tareas 5 · mezcla 13,537 B
ETL-parquet     presion  archivos leidos  120 · bytes leidos   2,281,385 · escritos     6,238 B en 1 archivo(s) · etapas 2 · tareas 5 · mezcla 30,133 B


In [26]:
# N2.6 Los dos productos ELT con sus SQL (dos corridas cada uno: la segunda confirma que el plan normalizado no cambia)
hip = json.loads((RAIZ / "hipotesis_s18.json").read_text(encoding="utf-8"))
for repeticion in range(2):
    for producto, clave in (("consumo", "elt_consumo"), ("presion", "elt_presion")):
        l = E.construir_y_registrar(spark, RAIZ, LAGO2, producto, EQUIPO, VOL2, BITACORA, hipotesis=hip[clave])
mostrar(ultima(VOL2, accion="construir", enfoque="ELT", producto="consumo"))
mostrar(ultima(VOL2, accion="construir", enfoque="ELT", producto="presion"))

ELT             consumo  archivos leidos  120 · bytes leidos   1,382,744 · escritos   275,768 B en 120 archivo(s) · etapas 3 · tareas 20 · mezcla 56,281 B
ELT             presion  archivos leidos  120 · bytes leidos     401,505 · escritos   253,897 B en 120 archivo(s) · etapas 3 · tareas 20 · mezcla 117,378 B


In [27]:
# N2.6 Sus hipotesis frente a lo observado (la diferencia es el dato que explican en la evidencia)
for producto, clave in (("consumo", "elt_consumo"), ("presion", "elt_presion")):
    b = ultima(VOL2, accion="construir", enfoque="ELT", producto=producto)
    print(f"hipotesis {clave}: esperaba {hip[clave]['valor']:,} · observado {b['bytes_leidos']:,} · diferencia {b['bytes_leidos'] - hip[clave]['valor']:+,} B (columnas leidas: {b['columnas_leidas']})")

hipotesis elt_consumo: esperaba 1,380,000 · observado 1,382,744 · diferencia +2,744 B (columnas leidas: ['sector', 'consumo_m3', 'quality_flag', 'fecha'])
hipotesis elt_presion: esperaba 400,000 · observado 401,505 · diferencia +1,505 B (columnas leidas: ['sector', 'quality_flag', 'fecha', 'nivel_presion'])


In [28]:
# N2.6 Resultado identico: el ELT filtrado reproduce al ETL (consumo y presion) y el ELT sin filtro reconcilia con la referencia
elt_c = E.leer_producto_pandas(LAGO2 / "consolidada_elt" / "consumo_dia_sector")
elt_p = E.leer_producto_pandas(LAGO2 / "consolidada_elt" / "presion_dia_sector")
etl_c, etl_p = leer_salida_etl(LAGO2, "consumo"), leer_salida_etl(LAGO2, "presion")
m = etl_c.merge(elt_c, on=["fecha", "sector"], suffixes=("_etl", "_elt"))
mp = etl_p.merge(elt_p, on=["fecha", "sector", "nivel_presion"], suffixes=("_etl", "_elt"))
ref = totales[["fecha", "sector", "consumo_total_m3", "n_lecturas"]].copy(); ref["fecha"] = ref["fecha"].astype(str)
mr = elt_c.merge(ref, on=["fecha", "sector"], suffixes=("", "_ref"))
print(f"consumo: {len(m)} grupos · conteos distintos {int((m.n_lecturas_etl != m.n_confiables).sum())} · diferencia maxima {(m.consumo_total_m3_etl - m.consumo_confiable_m3).abs().max():.1e}")
print(f"presion: {len(mp)} grupos · conteos distintos {int((mp.n_lecturas_etl != mp.n_confiables).sum())}")
print(f"sin filtro frente a la referencia: {len(mr)} grupos · diferencia maxima {(mr.consumo_total_m3 - mr.consumo_total_m3_ref).abs().max():.1e}")
print(f"lecturas guardadas {int(elt_c.n_lecturas.sum()):,} · confiables {int(elt_c.n_confiables.sum()):,} · descartadas por el ETL {int(elt_c.n_no_confiables.sum()):,} ({elt_c.n_no_confiables.sum() / elt_c.n_lecturas.sum() * 100:.2f} %)")
assert len(m) == len(elt_c) == len(mr) and (m.n_lecturas_etl == m.n_confiables).all() and (mp.n_lecturas_etl == mp.n_confiables).all()
assert (m.consumo_total_m3_etl - m.consumo_confiable_m3).abs().max() <= 1e-6 and (mr.consumo_total_m3 - mr.consumo_total_m3_ref).abs().max() <= 1e-6
assert int(elt_c.n_lecturas.sum()) == int(totales.n_lecturas.sum())

consumo: 480 grupos · conteos distintos 0 · diferencia maxima 0.0e+00
presion: 1440 grupos · conteos distintos 0
sin filtro frente a la referencia: 480 grupos · diferencia maxima 2.2e-12
lecturas guardadas 1,002,240 · confiables 942,031 · descartadas por el ETL 60,209 (6.01 %)


### N2.7 La copia alterada de su ampliación

La celda inyecta la alteración en su día de mayor consumo, reconstruye la preparación de la copia (`refinada_spark.main` cierra la sesión al terminar, por eso se abre de nuevo), ejecuta el ETL sobre la copia y evalúa el contrato. Debe terminar con la imagen de siempre: el ETL no avisa y el contrato falla en la frontera cruda a preparación.

In [29]:
# N2.7 La copia alterada de su ampliacion: inyectar, reconstruir su preparacion (cierra la sesion al terminar), ETL sobre la copia y contrato
dia_alterado = E.P.dia_de_a(totales)
carpeta = LAGO2.name
subprocess.run([sys.executable, str(RAIZ / "practica" / "inyectar_deriva.py"), "--raiz", str(RAIZ), "--carpeta", carpeta, "--dia", dia_alterado], check=True, capture_output=True)
DERIVA = RAIZ / "_deriva"
import contextlib, io
captura = io.StringIO()                       # main imprime su linea de bitacora completa como JSON: se captura y se muestra lo esencial
with contextlib.redirect_stdout(captura):
    codigo = R.main(["--raiz", str(RAIZ), "--carpeta", str(DERIVA), "--salida", str(DERIVA / "refinada_spark"), "--bitacora", "_deriva/bitacora_refinada_spark.jsonl", "--volumen", VOL2, "--equipo", EQUIPO])
texto_main = captura.getvalue()
reg = json.JSONDecoder().raw_decode(texto_main[texto_main.index("{"):])[0]
print(f"refinada_spark.main sobre la copia termina con codigo {codigo}: su reconciliacion contra el control falla a proposito")
print(f"   filas leidas {reg['filas_leidas']:,} · filas no conformes {reg['filas_no_conformes']:,} · nulos de consumo_m3 {reg['nulos_por_columna']['consumo_m3']:,}")
print(f"   contra la refinada heredada: {reg['reconciliacion_heredada']} ({reg['detalle_heredada']})")
print(f"   contra el control de la cruda: {reg['reconciliacion_control']} ({reg['detalle_control']})")
spark = E.abrir_sesion()                      # main cierra la sesion al terminar: se abre de nuevo
E.registrar_prep(spark, LAGO2)
correr_etl(DERIVA, VOL2, "--producto", "consumo", "--salida", str(DERIVA / "tablero_etl"), "--bitacora", "_deriva/bitacora_capas.jsonl")
real, copia = leer_salida_etl(LAGO2, "consumo"), leer_salida_etl(DERIVA, "consumo")
print(f"dia alterado {dia_alterado} · ETL sobre la copia, sin error: {len(copia)} filas (antes {len(real)}) · consumo {copia.consumo_total_m3.sum():,.3f} m3 (antes {real.consumo_total_m3.sum():,.3f})")
datos = E.tablas_cruda_a_preparacion(DERIVA / "refinada_spark", LAGO2 / "cruda" / "acueducto")
res, linea = E.evaluar_y_registrar_contrato(RAIZ, RAIZ / "contratos" / "cruda_a_preparacion.json", datos, "_deriva/refinada_spark", EQUIPO, VOL2, BITACORA)
nulos = int(datos["preparacion"]["consumo_m3"].isna().sum())
for r in res:
    print(f"   {r['invariante']} {r['tipo']:20s} {'ok' if r['ok'] else 'FALLA'}")
print(f"nulos de consumo_m3 en la preparacion de la copia: {nulos:,} · primer invariante que falla: {linea['invariante_que_falla']}")
assert not linea["aprobado"] and nulos > 0

refinada_spark.main sobre la copia termina con codigo 1: su reconciliacion contra el control falla a proposito
   filas leidas 1,002,240 · filas no conformes 8,352 · nulos de consumo_m3 8,352
   contra la refinada heredada: OK (1440 grupos iguales (tolerancia 1e-06); diferencia maxima de sumas 0.00e+00)
   contra el control de la cruda: FALLA (0 grupos con distinto numero de lecturas y 4 con distinta suma)


dia alterado 2024-01-14 · ETL sobre la copia, sin error: 480 filas (antes 480) · consumo 149,503.907 m3 (antes 150,767.171)


   C1 columnas_requeridas  ok
   C2 sin_nulos            FALLA
   C3 unico                ok
   C4 conteo_igual         ok
   C5 suma_igual           FALLA
   C6 igual_por_grupo      FALLA
   C7 valores_en           ok
nulos de consumo_m3 en la preparacion de la copia: 8,352 · primer invariante que falla: C2


### N2.8 Reproceso de la ventana del equipo

**Decisión de ingeniería de este nivel:** granularidad de la partición de salida (decisión 5): el valor inicial es por día. Si lo cambian, expliquen cuántos archivos y bytes se reescriben por ventana con cada diseño.

In [30]:
# N2.8 Reproceso de la ventana del equipo sobre una copia de la capa de salida: dinamica dos veces (idempotencia) y estatica
tabla = "consumo_dia_sector"
origen, copia = LAGO2 / "consolidada_elt" / tabla, RAIZ / "_copias_reproceso" / tabla
corridas = []
for modo, h in (("dinamica", None), ("dinamica", hip["reproceso"]), ("estatica", None)):
    E.copiar_salida(origen, copia)
    corridas.append(E.reprocesar_y_registrar(spark, RAIZ, LAGO2, "consumo", VENTANA2, EQUIPO, VOL2, copia, modo, BITACORA, hipotesis=h))
resumen = pd.DataFrame([{"sobrescritura": c["sobrescritura"], "intactas": c["particiones_intactas"], "reescritas": c["particiones_reescritas"],
                         "perdidas": c["particiones_perdidas"], "bytes de columnas": c["bytes_leidos"], "bytes escritos": c["bytes_escritos"], "hash": c["hash_resultado"][:10]} for c in corridas])
print(resumen.to_string(index=False))
total = corridas[0]["particiones_antes"]
assert corridas[0]["hash_resultado"] == corridas[1]["hash_resultado"]
assert (corridas[0]["particiones_reescritas"], corridas[0]["particiones_perdidas"], corridas[0]["particiones_intactas"]) == (dias_ventana, 0, total - dias_ventana)
assert corridas[2]["particiones_perdidas"] == total - dias_ventana

sobrescritura  intactas  reescritas  perdidas  bytes de columnas  bytes escritos       hash
     dinamica       114           6         0              69278           13811 00fc866148
     dinamica       114           6         0              69278           13811 00fc866148
     estatica         0           6       114              69278           13811 67920bd2cd


### N2.9 a N2.11 Entregables y verificación

Texto: documento de decisión, costo, ficha de la capa y glosario, a partir de las plantillas de `plantillas/t18/`. Luego la etapa `t18` imprime 18 comprobaciones numeradas con el valor observado frente al esperado.

In [31]:
# N2.9 Los entregables de texto: copien las plantillas y completenlas; la fila t18 del indice se agrega aqui
dest = RAIZ / "entregables" / "t18"
dest.mkdir(parents=True, exist_ok=True)
for nombre in ("decisiones.md", "costo.md", "ficha_capa.md", "glosario.md"):
    if not (dest / nombre).exists():
        shutil.copyfile(RAIZ / "plantillas" / "t18" / nombre, dest / nombre)
fila = "| t18 · ETL frente a ELT | Productos ELT idénticos al ETL, contratos por frontera, reproceso por ventana y entregables | `docker compose --profile s18 exec spark python pipeline/verificar_etapa.py --etapa t18` | `Etapa t18: APROBADA (18 de 18 comprobaciones, N s)` | (su commit) | (pegue aquí la salida) | (su equipo) |"
estado = E.registrar_fila_indice(RAIZ / "indice_evidencias.md", fila)      # la fila queda DENTRO de la tabla, no suelta al final
assert V.fila_en_tabla((RAIZ / "indice_evidencias.md").read_text(encoding="utf-8"), "t18"), "la fila t18 debe quedar dentro de la tabla del indice"
print(f"fila t18 del indice: {estado}")
print("Plantillas copiadas a entregables/t18/; completen decisiones, costo, ficha y glosario antes de la verificacion final")

fila t18 del indice: ya estaba
Plantillas copiadas a entregables/t18/; completen decisiones, costo, ficha y glosario antes de la verificacion final


In [32]:
# N2.10 Verificacion de la etapa t18: prueba de que el lago heredado no cambio y comprobaciones numeradas
despues = {"lago": V.hash_arbol_lago(LAGO2), "preparacion": V.hash_arbol_capa(LAGO2 / "refinada_spark")}
E.registrar_integridad(RAIZ, LAGO2, VOL2, HASHES2_ANTES, despues, EQUIPO, BITACORA)
E.escribir_comparacion(RAIZ, LAGO2, VOL2, RAIZ / "comparacion.csv", BITACORA)
r = subprocess.run([sys.executable, str(RAIZ / "pipeline" / "verificar_etapa.py"), "--etapa", "t18", "--volumen", VOL2], cwd=RAIZ, capture_output=True, text=True)
for linea in r.stdout.splitlines():
    print(linea[:230])
CODIGO_T18 = r.returncode

Verificando la etapa t18 en /proyecto

[APROBADO   ] pipeline/elt_s18.py, los SQL y los contratos sin NotImplementedError ni TODO; dos SQL en sql/consolidada y dos contratos: observado ([], 2, 2) · esperado ([], 2, 2)
[APROBADO   ] bitacora_capas.jsonl: ETL (consumo, presion, una pasada, ventana), ELT (consumo, presion), reproceso y contrato con todos los campos: observado [] · esperado []
[APROBADO   ] hipotesis numerica registrada antes de medir (bytes de columnas que leera el producto ELT de consumo, bytes de columnas que leera el producto ELT de presion, particiones que reescribira el reproceso): observado True 
[APROBADO   ] consolidada_elt/consumo_dia_sector: filas del grano de la referencia, clave (fecha, sector) unica y rutas con ceros: observado (480, 0, True) · esperado (480, 0, True)
[APROBADO   ] sin filtro: suma de n_lecturas igual a las filas de refinada_spark/ y a la referencia; diferencia maxima de consumo: observado (1002240, 480, '2.2e-12') · esperado (1002240, 480, '

In [33]:
# Autoverificacion final del Nivel 2
assert CODIGO_T18 == 0, "La etapa t18 no esta aprobada: lean las comprobaciones que dicen NO APROBADO"
print("Nivel 2 completo: etapa t18 aprobada")

Nivel 2 completo: etapa t18 aprobada


In [34]:
# N2.11 evidencia_nivel2_<equipo>.md prellenada: encabezado, hipotesis frente a lo observado y tabla de metricas
b = {(x["producto"], x["enfoque"]): x for x in E.leer_bitacora(RAIZ, VOL2, BITACORA) if x.get("accion") == "construir"}
tabla = E.escribir_comparacion(RAIZ, LAGO2, VOL2, RAIZ / "comparacion.csv", BITACORA)
texto = f"""# Evidencia del Nivel 2 · {EQUIPO}

1. **Encabezado.** Volumen {VOL2} ({LAGO2.name}) · ventana {VENTANA2[0]} a {VENTANA2[1]} · Spark {spark.version} · adaptativa apagada · 8 particiones de mezcla · metrica de bytes `bytes_columnas_leidas`.
2. **Hipótesis (con marca de tiempo).** {json.dumps({k: [h['valor'], h['marca_hipotesis']] for k, h in hip.items()}, ensure_ascii=False)}
3. **Métricas observadas.**

{a_markdown(tabla[['producto', 'enfoque', 'archivos_leidos', 'bytes_leidos', 'bytes_escritos', 'filas_descartadas_visibles']])}

4. **Hipótesis frente a lo observado.** _(una frase por hipótesis)_
5. **Decisiones del nivel y la cifra que las cambiaría.** _(granularidad, invariantes, qué va por ETL)_
6. **Conclusión** que separa lo observado de lo interpretado. _(su texto)_
"""
(RAIZ / f"evidencia_nivel2_{EQUIPO}.md").write_text(texto, encoding="utf-8")
print("escrito evidencia_nivel2_" + EQUIPO + ".md")

escrito evidencia_nivel2_equipo-1.md


## Nivel 3 · Frontera (opcional, fuera de las 3 horas)

**3a · Qué no debe aterrizar (decisión 8).** La celda da las cifras; ustedes deciden y documentan: descartar en el borde, mandar a cuarentena o retener con bandera, con el costo de lamentarse (recuperar lo descartado exige releer la cruda) frente al de retener.

| Decisión | Opciones | Criterios | Consecuencia de decidir mal | Qué debe justificarse |
|---|---|---|---|---|
| Qué se hace con las lecturas `SIN_SENAL` | Descartar en el borde · cuarentena aparte · retener con bandera | Costo de necesitarlas después, bytes retenidos, norma aplicable | Perder evidencia, o retener lo que no debía aterrizar | Bytes retenidos por año con supuestos declarados, costo de recuperar y la alternativa adoptada |

In [35]:
# Nivel 3a: que no debe aterrizar. Las cifras para el registro de decision (decision 8)
calidad = pd.concat([pq.read_table(f, columns=["quality_flag"]).to_pandas() for f in archivos2])["quality_flag"].value_counts()
total = int(calidad.sum())
por_dia = M.tamano_de(archivos2) / len(archivos2)
print((calidad / total * 100).round(2).to_string())
print(f"SIN_SENAL: {int(calidad.get('SIN_SENAL', 0)):,} de {total:,} lecturas ({calidad.get('SIN_SENAL', 0) / total * 100:.2f} %)")
print(f"retener toda la preparacion cuesta {por_dia:,.0f} B por dia; un anio de 365 dias son {por_dia * 365 / 2**20:.2f} MiB")
print(f"recuperar lo descartado por el ETL cuesta releer la cruda: {ultima(VOL2, accion='construir', enfoque='ETL', producto='consumo')['bytes_leidos']:,} B")

quality_flag
OK           93.99
ESTIMADO      5.01
SIN_SENAL     1.00
SIN_SENAL: 10,003 de 1,002,240 lecturas (1.00 %)
retener toda la preparacion cuesta 37,661 B por dia; un anio de 365 dias son 13.11 MiB
recuperar lo descartado por el ETL cuesta releer la cruda: 54,371,830 B


**Su decisión · Nivel 3a (decisión 8).** **Retener con bandera** las lecturas `SIN_SENAL`: son el **1,00 %** (10 003 de 1 002 240). Retener toda la preparación cuesta **37 661 B por día** (13,11 MiB al año), y recuperar lo descartado por el ETL obligaría a releer la cruda completa (**54 371 830 B**). Descartarlas en el borde ahorraría unos 130 KiB al año y perdería la evidencia de qué medidores no reportaron.

**3b · dbt-core (opcional).** El mismo producto como modelo SQL con `ref()` y pruebas, con DuckDB sobre los mismos Parquet, sin servicios nuevos. Requiere la capa opcional de dbt de la imagen (la guía explica cómo construirla y qué hacer sin conexión); si no está, la celda lo avisa y se omite. Si más adelante ejecutan `docker compose build`, la imagen se reconstruye sin esa capa y habrá que construirla de nuevo.

| Decisión | Opciones | Criterios | Consecuencia de decidir mal | Qué debe justificarse |
|---|---|---|---|---|
| Cómo se expresa y se versiona la transformación (decisión 7) | SQL con `spark.sql` y archivos · DataFrame · función propia · modelo dbt | Legibilidad para quien consume, plan resultante, versionado, dependencias nuevas | Lógica dispersa en cuadernos sin versión, o una herramienta que se vuelve un desvío | Mismo `hash_resultado` que el producto de Spark, dos pruebas que fallan ante la alteración y qué cifras no se trasladan (los bytes de Spark no son los de DuckDB) |

In [36]:
# Nivel 3b (opcional): el mismo producto como modelo de dbt sobre los Parquet, con DuckDB. Requiere la imagen con la capa de dbt
if shutil.which("dbt") is None:
    print("dbt no esta instalado en la imagen. Construyan la capa opcional de dbt (ver la guia) y reinicien el servicio.")
    DBT_OK = False
else:
    DBT = RAIZ / "dbt_acueducto"
    (RAIZ / "_dbt").mkdir(exist_ok=True)
    entorno = dict(os.environ, LAGO=str(LAGO2), DBT_TARGET_PATH="/tmp/dbt_target", DBT_LOG_PATH="/tmp/dbt_logs", DBT_USE_COLORS="false", NO_COLOR="1")
    r = subprocess.run(["dbt", "build", "--project-dir", str(DBT), "--profiles-dir", str(DBT)], env=entorno, capture_output=True, text=True)
    limpio = re.sub(chr(27) + r"\[[0-9;]*m", "", r.stdout)       # sin colores ni secuencias de escape
    print(chr(10).join(l for l in limpio.splitlines() if "PASS" in l or "FAIL" in l or "Done" in l)[:1500])
    h_dbt = E.hash_salida(RAIZ / "_dbt" / "consumo_dia_sector", ["fecha", "sector"])
    h_spark = ultima(VOL2, accion="construir", enfoque="ELT", producto="consumo")["hash_resultado"]
    print("hash dbt  :", h_dbt[:16], "\nhash Spark:", h_spark[:16], "\niguales   :", h_dbt == h_spark)
    DBT_OK = h_dbt == h_spark
    assert DBT_OK

dbt no esta instalado en la imagen. Construyan la capa opcional de dbt (ver la guia) y reinicien el servicio.


In [37]:
# Nivel 3b: las pruebas de dbt frente a la copia alterada (deben fallar al menos dos). Se ejecutan el modelo y las pruebas por separado
# para que una prueba que falla no impida correr las demas.
if shutil.which("dbt") is None:
    print("dbt no esta instalado: celda omitida")
else:
    entorno = dict(os.environ, LAGO=str(RAIZ / "_deriva"), DBT_TARGET_PATH="/tmp/dbt_target", DBT_LOG_PATH="/tmp/dbt_logs", DBT_USE_COLORS="false", NO_COLOR="1")
    comun = ["--project-dir", str(DBT), "--profiles-dir", str(DBT)]
    subprocess.run(["dbt", "run"] + comun, env=entorno, capture_output=True, text=True)
    r = subprocess.run(["dbt", "test"] + comun, env=entorno, capture_output=True, text=True)
    limpio = re.sub(chr(27) + r"\[[0-9;]*m", "", r.stdout)
    lineas = [l.split("  ", 1)[-1] for l in limpio.splitlines() if " PASS " in l or " FAIL " in l or "Done" in l]
    print(chr(10).join(l[:150] for l in lineas))
    assert sum(" FAIL " in l for l in lineas) >= 2, "al menos dos pruebas deben fallar ante la alteracion"

dbt no esta instalado: celda omitida


## Reto de diseño · Ficha de la capa (30 minutos, opcional)

**Situación.** La dirección financiera y operaciones preguntan por qué el tablero dice una cifra y la capa otra. Escriban la ficha de la capa: una página, sin jerga, con las seis secciones de `plantillas/t18/ficha_capa.md`. Competencia: **Power Humanise** (comunicar una decisión técnica y sus garantías con claridad y honestidad sobre sus límites).

| Decisión | Opciones | Criterios | Consecuencia de decidir mal | Qué debe justificarse |
|---|---|---|---|---|
| Cómo explicar dos cifras a quien no es ingeniero(a) | Un número con nota al pie · dos cifras con su diferencia explicada · un tablero con la bandera visible | Quién decide con la cifra, qué malentiende si lee solo una, cuánto cuesta pedir un recálculo | Que se confíe en una cifra sin saber qué dejó fuera, o que se pierda confianza en toda la capa | Las cifras de su bitácora (guardadas, contadas, retenidas), qué garantiza el contrato y qué no, y cómo se pide un recálculo |

In [38]:
# Reto de diseno: las cifras para la ficha de la capa (una pagina, sin jerga, para la direccion financiera y operaciones)
elt_c_df = E.leer_producto_pandas(LAGO2 / "consolidada_elt" / "consumo_dia_sector")
tot, conf = int(elt_c_df.n_lecturas.sum()), int(elt_c_df.n_confiables.sum())
print(f"lecturas guardadas {tot:,} · contadas por el tablero {conf:,} · retenidas con bandera {tot - conf:,} ({(tot - conf) / tot * 100:.2f} %)")
print(f"consumo guardado {elt_c_df.consumo_total_m3.sum():,.1f} m3 · consumo del tablero {elt_c_df.consumo_confiable_m3.sum():,.1f} m3")
print("Escriban la ficha en entregables/t18/ficha_capa.md con los seis apartados de la plantilla (150 a 450 palabras).")

lecturas guardadas 1,002,240 · contadas por el tablero 942,031 · retenidas con bandera 60,209 (6.01 %)
consumo guardado 160,402.5 m3 · consumo del tablero 150,767.2 m3
Escriban la ficha en entregables/t18/ficha_capa.md con los seis apartados de la plantilla (150 a 450 palabras).


**Su decisión · Reto.** Mostramos **las dos cifras lado a lado, con su diferencia explicada**: lecturas guardadas (1 002 240) frente a contadas por el tablero (942 031), y consumo total (160 402 m³) frente a confiable (150 767 m³). No escondemos la cifra menor ni ponemos una nota al pie: así la diferencia se entiende como lecturas marcadas, no como un error. Primer párrafo de la ficha:

> *Guardamos 1 002 240 lecturas. El tablero cuenta 942 031, las confiables. Las otras 60 209 (6,0 %) no se borran: quedan guardadas y marcadas. Por eso hay dos consumos: 160 402 m³ con todas las lecturas y 150 767 m³ con las confiables, que es la cifra del tablero.*

## Lista de entrega y autoverificación

- [ ] `pipeline/elt_s18.py` completado, `sql/consolidada/*.sql` (dos productos), `contratos/*.json` (dos fronteras), `hipotesis_s18.json` y `bitacora_capas.jsonl`.
- [ ] `comparacion.csv`, `planes_s18/` y `evidencia_nivel2_<equipo>.md`.
- [ ] `entregables/t18/decisiones.md`, `costo.md`, `ficha_capa.md` y `glosario.md`, y la fila `t18` de `indice_evidencias.md` con la salida de la etapa.
- [ ] Versionados también: `docker-compose.yml`, `perfiles/s15/spark/Dockerfile` (y `requirements_dbt.txt` si hicieron el Nivel 3b) y **este cuaderno ejecutado** (con sus salidas).
- [ ] La salida de `python pipeline/verificar_etapa.py --etapa t18` dice `APROBADA (18 de 18 comprobaciones)`.
- [ ] Otra persona repitió un producto con solo el `README.md` y obtuvo el mismo `hash_resultado`.

In [39]:
# Cierre: cierren la sesion de Spark (el cuaderno ejecutado, la bitacora y los entregables se versionan en el repositorio del equipo)
spark.stop()
print("sesion cerrada")

sesion cerrada
